# ThoughtZero: Kaggle GPU server template

Copy this notebook, then set **Settings → Accelerator → GPU T4 x2** and **Internet on**.

Add a Kaggle *secret* (Add-ons → Secrets) with the key for the Jev route set in `JEV_TRANSPORT` below: `OPENROUTER_API_KEY`, `TYPESAFE_API_KEY`, `OPENCODE_ZEN_API_KEY`, `AI_GATEWAY_API_KEY` or `BOCHA_API_KEY` (docs/verified_apis.md, "Jev routes"). Only runs that call Jev need it.
Never paste a key into a cell. **Clear all outputs before sharing.**

Status: draft. fp16 correctness on a T4 is unverified (docs/gpu_setup.md).

In [ ]:
# --- Parameters ---------------------------------------------------------
# 4-bit QAT checkpoint: the bf16 E4B (16 GB) does not fit a T4
MODEL = "google/gemma-4-E4B-it-qat-w4a16-ct"
TOKENIZER = "google/gemma-4-E4B-it"
REPO = "https://github.com/AkulRanjan/pyschic-vla.git"
BRANCH = "main"
JEV_TRANSPORT = "openrouter"  # direct | openrouter | zen | vercel | bocha
# What to run once the server is up. The real smoke test (Gemma here + Jev) is
#   "scripts/smoke_test.py --n-sims 4 --k 3 --tokenizer google/gemma-4-E4B-it"
SCRIPT = "scripts/smoke_test.py --mock"

In [ ]:
!pip install -q -U "vllm>=0.19.1"  # TODO: pin the exact version once verified on a T4

In [ ]:
import os
import subprocess
import time
import urllib.request

env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")  # GPU 1 stays free for the PRM
cmd = [
    "vllm",
    "serve",
    MODEL,
    "--dtype",
    "float16",
    "--max-model-len",
    "8192",
    "--enable-prefix-caching",
    "--max-logprobs",
    "20",
    "--host",
    "127.0.0.1",
    "--port",
    "8000",
]
log_file = open("vllm.log", "w")  # noqa: SIM115 (must outlive this cell)
server = subprocess.Popen(cmd, env=env, stdout=log_file, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2)
        print("server up")
        break
    except Exception as err:
        if server.poll() is not None:
            raise RuntimeError("vLLM exited; see vllm.log") from err
        time.sleep(5)
else:
    raise TimeoutError("server did not come up; see vllm.log")

In [ ]:
# Sanity check: one greedy completion. Look for garbage/NaN output (fp16 overflow).
import json

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TOKENIZER)
prompt = tok.apply_chat_template(
    [{"role": "user", "content": "What is 2+3? Answer in \\boxed{}."}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
body = {"model": MODEL, "prompt": prompt, "max_tokens": 64, "temperature": 0, "logprobs": 5}
req = urllib.request.Request(
    "http://127.0.0.1:8000/v1/completions",
    data=json.dumps(body).encode(),
    headers={"Content-Type": "application/json"},
)
print(json.load(urllib.request.urlopen(req))["choices"][0]["text"])

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ""


subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, "repo"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "repo"], check=True)
with open("repo/.env", "w") as f:
    f.write(f"GEMMA_BASE_URL=http://127.0.0.1:8000/v1\nGEMMA_MODEL={MODEL}\n")
    f.write(f"JEV_TRANSPORT={JEV_TRANSPORT}\n")
    for name in (
        "TYPESAFE_API_KEY",
        "OPENROUTER_API_KEY",
        "OPENCODE_ZEN_API_KEY",
        "AI_GATEWAY_API_KEY",
        "BOCHA_API_KEY",
    ):
        value = secret(name)
        if value:
            f.write(f"{name}={value}\n")

In [ ]:
!cd repo && python {SCRIPT}

In [ ]:
# Copy results to the notebook output, then stop the server.
!mkdir -p /kaggle/working/results
!cp -r repo/results/* /kaggle/working/results/ 2>/dev/null || true
server.terminate()